## Importing Data and setting up workbook etc

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.model_selection
import tensorflow as tf
from tensorflow.keras import activations
from keras.models import Sequential
from keras.layers import (Dense, Conv1D, LSTM, MaxPooling1D, Dropout,
                           Reshape, Flatten, BatchNormalization, Activation)
from keras.regularizers import l2
%matplotlib inline

from neural_seismic import import_traces, mean_residual_calcs
from neural_seismic.picking import test_coppens, confidence_calcs, convert_series
from neural_seismic.utils import notify

In [ ]:
# Here we import from a specifcally formatted csv. 
comp_trace = import_traces(4000,'Rio')
for trace in comp_trace:
    if pd.Series(trace.FB_Picks[0]).isnull().any():
        trace.FB_Picks[0]=0
    trace.calc_metrics()
    trace.gen_feat_space() # i've modified this to normalise the feature space
    trace.feat_space.fillna(0)

In [ ]:
# some of our samples have no first break pick, this will cause issues later so we simply remove them from our analysis for now.
no_zeros=[x for x in comp_trace if x.FB_Picks[1]!=0]
no_zeros=[x for x in no_zeros if x.FB_Picks[0]!=0]
print(len(no_zeros))
small=no_zeros[:3000]

In [ ]:
check=[0]
while len(check)<3:
    train, test = sklearn.model_selection.train_test_split(small,train_size=0.6)
    train, vali = sklearn.model_selection.train_test_split(train,train_size=1200)
    check = [x for x in test if x.iD in [88,151,103]]#,2033,4216]] # this check ensures that all our key examples are in our predicted set while keeping random 

In [ ]:
# Generator for series vale prediction
def series_inputs(batch_T_obj,feat_num,samples_per_epoch=None,batch_size=1):
    """
    Our series_ generator creates input sets for our model training
    batch_T_obj
    feat_num
    """
    if samples_per_epoch==None:
        samples_per_epoch=len(batch_T_obj)

    counter=0
    while counter < len(batch_T_obj):
        #print([trace.iD for trace in batch_T_obj[counter:counter+batch_size]])
        target = np.stack([trace.feat_space.values[:,35].reshape((500,1)).astype(np.float32) for trace in batch_T_obj[counter:counter+batch_size]])
        signal = np.stack([trace.feat_space.values[:,:feat_num].reshape((500,feat_num)).astype(np.float32) for trace in batch_T_obj[counter:counter+batch_size]])
        counter += batch_size
        yield(signal,target)
    
    if counter >= samples_per_epoch:
        counter=0

class MCDropout(tf.keras.layers.Layer):
    def __init__(self,rate):
        super(MCDropout,self).__init__()
        self.rate = rate
    def call(self,inputs):
        return(tf.nn.dropout(inputs,rate=self.rate))

## Building Models in Keras
In my work I use keras to build sequential models. These sequential models all follow the same broad process and are handled by various build functions.
These build functions of the various models are below and variations as suggested by the reviewer are tested. 

---

**These include the impact on model performace cause by: **
- Actvation functions

- Batch Normilisation 
    - Originally I had a batch size of 1 and normilised each trace
    - Model modifications now allow for simple choice between generating pre normilised sequences or normilising them batch by batch in the model

- Model Regularisation

**Further that the models can be better understood using: **

- Improved Uncertainty Estimate using MC dropout

---

The functions below can now generate and test these suggested variants for the BPNN and CONV models as requested. 
If all values are left as defaults then the original models I tested are returned. Model by model and parameter by parameter we'll test to see technical improvedments when trained over 600 samples, and predicting over 600.



In [ ]:
def BPNN_revised(   hidden_neuron=10, n_hidden=10, optim='Adamax', a_func=None, num_feat=1, 
                    MCD=False, batch_norm_input=False, batch_norm_post=False, l2_reg_factor=None ):
    """
    The 'simple' back propigated neural network or dense neural network of my paper
    
    Arguments:

    hidden_neuron (int)             :   number of neurons in each hidden layer
    n_hidden (int)                  :   number of hidden layers in our model
    optim (str)                     :   string from keras optimisers
    a_func (a_func)                 :   activation function for the hidden layers if None defaults to linear
    batch_size (int)                :   number of samples in each training batch defaults to 1
    num_feat (int)                  :   number of features at each time step defaults to 1
    batch_norm_inputs(bool)         :   batch normilise the inputs, if batch_size set to 1 normilise a single sample
    batch_norm_post(bool)           :   do we batch normilise the outputs of each hidden layer (before activation function)
    l2_reg_factor(float)            :   if not none we perform l2 regularisation on the hidden state using the float as the l2 regulisation factor
    
    Returns:

    modelexp (keras sequential)     :  for training and predicting.

    """

                                                                                                    # 0.0 - make a sequential model instance
    modelexp = Sequential()
                                                                                                    # 1.0 - lets define the first layer which either 
    if batch_norm_input:                                                                            # 1.1 - batch normalises the data
        modelexp.add(BatchNormalization(input_shape=(500, num_feat)))
        modelexp.add(Dense(50,activation=None))
                                                      
    if not batch_norm_input:                                                                        # 1.1 - alternatively a standard dense input
        modelexp.add(Dense(50, input_shape=(500, num_feat),activation=None))

    if MCD:
        model.exp.add(MCDropout(rate=0.5))

                                                                                                    # 2.0 - the hidden part!
    for i in range(n_hidden):                                                                       # 2.1 - number of hidden layers
        reg = None
        if not l2_reg_factor==None:                                                                 # 2.2 - will we do l2 norm regulisation
            reg = l2(l2=l2_reg_factor)                                                              # 2.3 - if so lets redefine it and set its regularisation factor

        if batch_norm_post:                                                                         # 2.4 - what about second batch normilization
            modelexp.add(Dense(hidden_neuron,kernel_regularizer=reg))                               # 2.5 - add our dense layer and specify our kernel_regularizer
            modelexp.add(BatchNormalization())                                                      # 2.6 - put the batch norm between the layer and the activation
            modelexp.add(Activation(a_func))                                                        # 2.7 - don't forget that activation function

        if not batch_norm_post:                                                                     # 2.4 - alternatively if not batch_norm_post lets just add the layer
            modelexp.add(Dense(hidden_neuron,activation=a_func,kernel_regularizer=reg))             # 2.5 - boop dense layer what fun

    modelexp.add(Dense(1))                                                                          # 3.0 - Boom dense layer doing the good stuff
    modelexp.compile(loss='mean_absolute_error',optimizer=optim,metrics=['accuracy'])               # 3.1 - compile the model and lets get out of here
    
    return(modelexp)

In [ ]:
def CONV_revised(   hidden_neuron=10, hid_units = 2, optim='Adamax', a_func=None,
                    num_feat=1, batch_norm_input=False, batch_norm_post=False, l2_reg_factor=None,MCD=False ):
    """
    The convolutional network instead of layerse uses units. 
    Each unit containts a convolutional layer a pooling layer followed by a dense layer which reshapes the outputs.
    
    Arguments:

    hidden_neuron (int)             :   number of neurons in each hidden layer
    n_hidden (int)                  :   number of hidden layers in our model
    optim (str)                     :   string from keras optimisers
    a_func (a_func)                 :   activation function for the hidden layers if None defaults to linear
    batch_size (int)                :   number of samples in each training batch defaults to 1
    num_feat (int)                  :   number of features at each time step defaults to 1
    batch_norm_inputs(bool)         :   batch normilise the inputs, if batch_size set to 1 normilise a single sample
    batch_norm_post(bool)           :   do we batch normilise the outputs of each hidden layer (before activation function)
    l2_reg_factor(float)            :   if not none we perform l2 regularisation on the hidden state using the float as the l2 regulisation factor
    
    Returns:

    modelexp (keras sequential)     :  for training and predicting.

    """

                                                                                                    # 0.0 - make a sequential model instance
    modelexp = Sequential()
                                                                                                    # 1.0 - lets define the first unit which either 
    if batch_norm_input:                                                                            # 1.1 - batch normalises the data
        modelexp.add(BatchNormalization(input_shape=(500, num_feat)))
        modelexp.add(Conv1D(hidden_neuron,kernel_size=20,padding="same",activation=a_func))
        modelexp.add(MaxPooling1D(pool_size=20))
        modelexp.add(Dense(20))
        modelexp.add(Flatten())
        modelexp.add(Reshape((500,1)))
                                                      
    if not batch_norm_input:                                                                        # 1.1 - alternatively a standard unit non-normilised input
        modelexp.add(Conv1D(hidden_neuron,input_shape=(500, num_feat),kernel_size=20,padding="same",activation=a_func))
        modelexp.add(MaxPooling1D(pool_size=20))
        modelexp.add(Dense(20))
        modelexp.add(Flatten())
        modelexp.add(Reshape((500,1)))
    if MCD:
        modelexp.add(MCDropout(rate=0.5))
                                                                                        # 2.0 - the hidden part!
    for i in range(hid_units-1):                                                                       # 2.1 - number of hidden layers
        reg = None
        if not l2_reg_factor==None:                                                                 # 2.2 - will we do l2 norm regulisation
            reg = l2(l2=l2_reg_factor)                                                              # 2.3 - if so lets redefine it and set its regularisation factor

        if batch_norm_post:                                                                         # 2.4 - what about second batch normilization
            modelexp.add(Conv1D(hidden_neuron,input_shape=(500, num_feat),kernel_size=20,padding="same",kernel_regularizer=reg))
            modelexp.add(BatchNormalization())                                                      # 2.6 - put the batch norm between the layer and the activation
            modelexp.add(Activation(a_func))
            modelexp.add(MaxPooling1D(pool_size=20))
            modelexp.add(Dense(20))
            modelexp.add(Flatten())
            modelexp.add(Reshape((500,1)))


        if not batch_norm_post:                                                                     # 2.4 - alternatively if not batch_norm_post lets just add the layer
            modelexp.add(Conv1D(hidden_neuron,input_shape=(500, num_feat),activation=a_func,kernel_size=20,padding="same",kernel_regularizer=reg))
            modelexp.add(MaxPooling1D(pool_size=20))
            modelexp.add(Dense(20))
            modelexp.add(Flatten())
            modelexp.add(Reshape((500,1)))

    modelexp.add(Dense(1))                                                                          # 3.0 - Boom dense layer doing the good stuff
    modelexp.compile(loss='mean_absolute_error',optimizer=optim,metrics=['accuracy'])               # 3.1 - compile the model and lets get out of here
    modelexp.summary()
    return(modelexp)

### Testing Activation Functions of Hidden Layers
Activation functions are typically non-linear, to allow deeper models to convert linear input signals to non-linear output signals.
In our original models we used simple linear activation which the reviewer questions as potentially impacting our model performance.
In this section we compare the activation functions baked into keras and compare their performance in our BPNN and CNN on a subset of samples.

In [ ]:
# for the sake of plotting we'll make a dictionary to make plotting easier
activ_func = {}
activ_func["ReLu"] = activations.relu
activ_func["TanH"] = activations.tanh
activ_func["Swish"] = activations.swish
activ_func["S_Sign"] = activations.softsign
activ_func["S_plus"] = activations.softplus
activ_func["S_max"] = activations.softmax
activ_func["Sigmoid"] = activations.sigmoid
activ_func["SeLu"] = activations.selu
activ_func["H_Sigmoid"] = activations.hard_sigmoid
activ_func["Linear"] = None

# here none is our original linear model

In [ ]:
CNnn_activation = []
for i in range(len(list(activ_func.keys()))):
    acf = activ_func[list(activ_func.keys())[i]]
    CNA_I = CONV_revised(a_func=acf,batch_norm_input=True,MCD=True)
    CNnn_activation.append(CNA_I)

nnmodel_hist = []
nnmodel_pred = []
for model in CNnn_activation:
    nos = len(train)
    b_s = 1
    nob = int(len(train)/b_s)
    epc = 100
    spe = int(nob/epc)
    print("Model Training")
    MI_hist = model.fit(series_inputs(train,1,batch_size=b_s),steps_per_epoch=100/b_s,epochs=int(len(train)/100))
    print("Model Prediction")
    MI_pred = model.predict(series_inputs(test,1),steps=len(test))
    nnmodel_hist.append(MI_hist)
    nnmodel_pred.append(MI_pred)
    


In [ ]:

## GOOD PLOT STUFF
fig, axs = plt.subplots(2, 1, constrained_layout=True,sharex=True,figsize=(10,10))
fig.suptitle('Comparison of Activation Functions', fontsize=16)
for i in range(len(nnmodel_hist)):
    axs[0].plot(nnmodel_hist[i].history['accuracy'],label=list(activ_func.keys())[i])

axs[0].set_title('BPNN')
axs[0].set_ylim([0.3,1])
#axs[0].set_xlabel('Iterations')
axs[0].set_ylabel('Accuracy')
axs[0].legend(loc=4)#,ncol=len(cnmodel_hist))
#for i in range(len(cnmodel_hist)):
#    axs[1].plot(cnmodel_hist[i].history['accuracy'],label=list(activ_func.keys())[i])
#
#axs[1].set_title('CNN')
#axs[1].set_xlabel('Iterations')
#axs[1].set_ylabel('Accuracy')
#axs[1].set_ylim([0.3,1])
#axs[1].legend(loc=4)#,ncol=len(cnmodel_hist))
#plt.savefig('Activation Function - Comparison.png')

#axs[1].legend()

## Impact of Dropout
